# 12 — Baseline model training

This rerun trains the existing LightGBM baseline using only prepared training data and evaluates it on validation data. The test split is deliberately not loaded or inspected.

### What the cell does
Imports the required libraries, defines the allowed train/validation inputs and output locations, and records an SHA-256 checksum for every input.

### Why it is necessary
Explicit paths prevent accidental test-set access, while checksums prove that model training did not alter prepared data.

### What to understand
The printed paths and hashes identify exactly which immutable train and validation artifacts the LightGBM rerun uses.

In [1]:
from pathlib import Path
import hashlib
import time
import warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve,
                             average_precision_score)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert ROOT.name == "AdoptAI_V1", f"Run inside AdoptAI_V1, not {ROOT}"

PREPROCESSED_DIR = ROOT / "data/modeling/preprocessed"
MODEL_DIR = ROOT / "models/baseline"
REPORT_DIR = ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures/baseline_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

input_paths = {
    "X_train_tree": PREPROCESSED_DIR / "X_train_tree.csv",
    "X_validation_tree": PREPROCESSED_DIR / "X_validation_tree.csv",
    "y_train": PREPROCESSED_DIR / "y_train.csv",
    "y_validation": PREPROCESSED_DIR / "y_validation.csv",
    "train_identifiers": PREPROCESSED_DIR / "train_identifiers.csv",
    "validation_identifiers": PREPROCESSED_DIR / "validation_identifiers.csv",
}
assert all(path.exists() for path in input_paths.values()), "One or more required inputs are missing."

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

input_hashes_before = {name: sha256_file(path) for name, path in input_paths.items()}
for name, path in input_paths.items():
    print(f"{name}: {path} | SHA-256={input_hashes_before[name]}")

X_train_tree: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/preprocessed/X_train_tree.csv | SHA-256=4fcc85a5b85270f40b3d17e172e3a2236f2c5bbdb7966e1edcb05535ed2c4057
X_validation_tree: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/preprocessed/X_validation_tree.csv | SHA-256=8d9ff9e7aa7ca2afedadb0401538f68d15e6e4fd68c48b5648935dce4ec59fee
y_train: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/preprocessed/y_train.csv | SHA-256=e7ba6fccd47388067e865470f4b3eb65ae661a54e4f21e16700d716e76595ce3
y_validation: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/preprocessed/y_validation.csv | SHA-256=f6089607c952c1eb631961cbee20fbc99471c43d90da3329c5945a5e65fd5b34
train_identifiers: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/preprocessed/train_identifiers.csv | SHA-256=8831610522694802bbe69656699dd92d140b860b718a3841e8231e7ee1e585fe
validation_identifiers: /Users/fatimazahranamaoui/Desktop/AD

### What the cell does
Loads only the permitted training and validation files, then validates row counts, schemas, labels, identifiers, missing values, and finite numeric values.

### Why it is necessary
A baseline comparison is trustworthy only when every model receives aligned, valid rows and equivalent feature schemas.

### What to understand
Successful assertions mean that the newly generated training and validation rows are ready for modeling without further cleaning.

In [2]:
X_train_tree = pd.read_csv(input_paths["X_train_tree"])
X_validation_tree = pd.read_csv(input_paths["X_validation_tree"])
y_train_frame = pd.read_csv(input_paths["y_train"])
y_validation_frame = pd.read_csv(input_paths["y_validation"])
train_identifiers = pd.read_csv(input_paths["train_identifiers"])
validation_identifiers = pd.read_csv(input_paths["validation_identifiers"])

TARGET = "slowdown_in_5min"
EXPECTED_TRAIN_ROWS = 82_860
EXPECTED_VALIDATION_ROWS = 22_212
assert y_train_frame.columns.tolist() == [TARGET]
assert y_validation_frame.columns.tolist() == [TARGET]
y_train = y_train_frame[TARGET].astype(int)
y_validation = y_validation_frame[TARGET].astype(int)

assert len(X_train_tree) == len(y_train) == len(train_identifiers) == EXPECTED_TRAIN_ROWS
assert len(X_validation_tree) == len(y_validation) == len(validation_identifiers) == EXPECTED_VALIDATION_ROWS
assert X_train_tree.columns.tolist() == X_validation_tree.columns.tolist()
assert set(y_train.unique()) == {0, 1} and set(y_validation.unique()) == {0, 1}

feature_frames = [X_train_tree, X_validation_tree]
assert all(not frame.isna().any().any() for frame in feature_frames)
assert all(np.isfinite(frame.to_numpy(dtype=float)).all() for frame in feature_frames)
assert TARGET not in X_train_tree.columns
print(f"Train rows: {len(y_train):,}; validation rows: {len(y_validation):,}")
print(f"Features per model matrix: {X_train_tree.shape[1]:,}")
print("All schemas, targets, identifiers, missing-value checks, and finite-value checks passed.")

Train rows: 82,860; validation rows: 22,212
Features per model matrix: 360
All schemas, targets, identifiers, missing-value checks, and finite-value checks passed.


### What the cell does
Summarizes positive and negative labels and calculates the accuracy achieved by always predicting the majority class.

### Why it is necessary
When the target is imbalanced, ordinary accuracy can look high even when a model rarely detects an upcoming slowdown.

### What to understand
Balanced accuracy, recall, F1, and precision–recall AUC are needed alongside accuracy because the class distribution is strongly imbalanced and differs between splits.

In [3]:
class_distribution = []
for split_name, labels in [("train", y_train), ("validation", y_validation)]:
    positive_rows = int(labels.sum())
    negative_rows = int((labels == 0).sum())
    class_distribution.append({
        "split": split_name,
        "total_rows": len(labels),
        "positive_rows": positive_rows,
        "negative_rows": negative_rows,
        "positive_rate": positive_rows / len(labels),
        "majority_class_accuracy": max(positive_rows, negative_rows) / len(labels),
    })
class_distribution = pd.DataFrame(class_distribution)
display(class_distribution.style.format({"positive_rate": "{:.2%}", "majority_class_accuracy": "{:.2%}"}))
print("Accuracy alone is misleading: a model can score highly by favoring class 0 while missing future slowdowns.")

,split,total_rows,positive_rows,negative_rows,positive_rate,majority_class_accuracy
0,train,82860,28198,54662,34.03%,65.97%
1,validation,22212,10913,11299,49.13%,50.87%


Accuracy alone is misleading: a model can score highly by favoring class 0 while missing future slowdowns.


### What the cell does
Defines and fits only the existing fixed LightGBM baseline configuration, while recording training time.

### Why it is necessary
This reproduces the requested baseline without rerunning unrelated model families or performing hyperparameter search. Class imbalance is derived only from training labels.

### What to understand
Validation rows are never used for fitting, and threshold 0.50 remains the provisional baseline decision rule.

In [4]:
negative_to_positive_ratio = float((y_train == 0).sum() / (y_train == 1).sum())
model_specs = {
    "LightGBM": {
        "model": LGBMClassifier(objective="binary", n_estimators=300, learning_rate=0.05,
                                num_leaves=31, scale_pos_weight=negative_to_positive_ratio,
                                random_state=42, n_jobs=-1, verbosity=-1),
        "preprocessing_type": "tree: median + indicators",
        "X_train": X_train_tree, "X_validation": X_validation_tree,
    }
}

trained_models, timing, training_failures = {}, {}, {}
for model_name, spec in model_specs.items():
    started = time.perf_counter()
    try:
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")
            spec["model"].fit(spec["X_train"], y_train)
        timing[model_name] = {"training_time_seconds": time.perf_counter() - started}
        trained_models[model_name] = spec["model"]
        print(f"{model_name}: trained in {timing[model_name]['training_time_seconds']:.3f}s; warnings={len(captured_warnings)}")
    except Exception as exc:
        training_failures[model_name] = f"{type(exc).__name__}: {exc}"
        print(f"{model_name}: FAILED — {training_failures[model_name]}")
assert trained_models, "No baseline model trained successfully."
print(f"Training imbalance ratio used by LightGBM: {negative_to_positive_ratio:.4f} negatives per positive.")

LightGBM: trained in 4.997s; warnings=0
Training imbalance ratio used by LightGBM: 1.9385 negatives per positive.


### What the cell does
Generates validation probabilities and 0.50-threshold predictions, measures prediction time, and calculates classification and ranking metrics.

### Why it is necessary
Multiple metrics reveal different failure modes: false alerts, missed slowdowns, class balance, and probability ranking quality.

### What to understand
PR-AUC is especially informative for the positive slowdown class; the confusion-matrix counts show the operational cost behind each score.

In [5]:
THRESHOLD = 0.50
validation_outputs, comparison_rows = {}, []
for model_name, model in trained_models.items():
    X_validation_for_model = model_specs[model_name]["X_validation"]
    started = time.perf_counter()
    probability = model.predict_proba(X_validation_for_model)[:, list(model.classes_).index(1)]
    prediction = (probability >= THRESHOLD).astype(int)
    prediction_time = time.perf_counter() - started
    tn, fp, fn, tp = confusion_matrix(y_validation, prediction, labels=[0, 1]).ravel()
    validation_outputs[model_name] = {"probability": probability, "prediction": prediction}
    comparison_rows.append({
        "model": model_name,
        "preprocessing_type": model_specs[model_name]["preprocessing_type"],
        "threshold": THRESHOLD,
        "accuracy": accuracy_score(y_validation, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_validation, prediction),
        "precision": precision_score(y_validation, prediction, zero_division=0),
        "recall": recall_score(y_validation, prediction, zero_division=0),
        "f1": f1_score(y_validation, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_validation, probability),
        "pr_auc": average_precision_score(y_validation, probability),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "training_time_seconds": timing[model_name]["training_time_seconds"],
        "prediction_time_seconds": prediction_time,
    })
baseline_comparison = (pd.DataFrame(comparison_rows)
                       .sort_values(["pr_auc", "f1", "recall"], ascending=False)
                       .reset_index(drop=True))
display(baseline_comparison.style.format({
    "accuracy": "{:.4f}", "balanced_accuracy": "{:.4f}", "precision": "{:.4f}",
    "recall": "{:.4f}", "f1": "{:.4f}", "roc_auc": "{:.4f}", "pr_auc": "{:.4f}",
    "training_time_seconds": "{:.3f}", "prediction_time_seconds": "{:.4f}"}))

,model,preprocessing_type,threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp,training_time_seconds,prediction_time_seconds
0,LightGBM,tree: median + indicators,0.500000,0.8991,0.8991,0.8954,0.8997,0.8975,0.9589,0.9690,10152,1147,1095,9818,4.997,0.0663


### What the cell does
Creates and saves the LightGBM validation confusion matrix.

### Why it is necessary
The four matrix cells distinguish correct decisions from missed future slowdowns and false alarms.

### What to understand
Rows are actual outcomes and columns are model predictions; the lower-left cell contains missed slowdowns and the upper-right contains false alerts.

In [6]:
confusion_figure_paths = []
for model_name, output in validation_outputs.items():
    matrix = confusion_matrix(y_validation, output["prediction"], labels=[0, 1])
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    image = ax.imshow(matrix, cmap="Blues")
    for row in range(2):
        for column in range(2):
            ax.text(column, row, f"{matrix[row, column]:,}", ha="center", va="center",
                    color="white" if matrix[row, column] > matrix.max() / 2 else "black", fontsize=13)
    ax.set_xticks([0, 1], labels=["Predicted 0", "Predicted 1"])
    ax.set_yticks([0, 1], labels=["Actual 0: no slowdown", "Actual 1: future slowdown"])
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Actual class")
    ax.set_title(f"{model_name} — validation confusion matrix")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    figure_path = FIGURE_DIR / f"{model_name.lower()}_confusion_matrix.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    confusion_figure_paths.append(figure_path)
print(f"Saved {len(confusion_figure_paths)} confusion matrices under {FIGURE_DIR}")

Saved 1 confusion matrices under /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/baseline_models


/var/folders/3h/tpfh764d3msgh0j4_bys88yr0000gn/T/ipykernel_38209/1986837254.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### What the cell does
Plots the LightGBM validation precision–recall and ROC curves and saves both figures.

### Why it is necessary
Curves compare probability rankings across all possible thresholds instead of judging only the provisional 0.50 cutoff.

### What to understand
A stronger curve stays toward the upper-right in precision–recall space and upper-left in ROC space; PR curves deserve extra attention for this imbalanced target.

In [7]:
fig, ax = plt.subplots(figsize=(8, 6))
for model_name, output in validation_outputs.items():
    precision_values, recall_values, _ = precision_recall_curve(y_validation, output["probability"])
    score = average_precision_score(y_validation, output["probability"])
    ax.plot(recall_values, precision_values, linewidth=2, label=f"{model_name} (PR-AUC={score:.3f})")
ax.axhline(y_validation.mean(), color="gray", linestyle="--", label=f"Positive-rate baseline ({y_validation.mean():.3f})")
ax.set(xlabel="Recall", ylabel="Precision", title="Validation precision–recall comparison", xlim=(0, 1), ylim=(0, 1.02))
ax.grid(alpha=0.25); ax.legend(loc="best", fontsize=8); fig.tight_layout()
pr_curve_path = FIGURE_DIR / "precision_recall_comparison.png"
fig.savefig(pr_curve_path, dpi=160, bbox_inches="tight"); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 6))
for model_name, output in validation_outputs.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(y_validation, output["probability"])
    score = roc_auc_score(y_validation, output["probability"])
    ax.plot(false_positive_rate, true_positive_rate, linewidth=2, label=f"{model_name} (ROC-AUC={score:.3f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random ranking")
ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title="Validation ROC comparison", xlim=(0, 1), ylim=(0, 1.02))
ax.grid(alpha=0.25); ax.legend(loc="lower right", fontsize=8); fig.tight_layout()
roc_curve_path = FIGURE_DIR / "roc_comparison.png"
fig.savefig(roc_curve_path, dpi=160, bbox_inches="tight"); plt.show(); plt.close(fig)
print(f"Saved: {pr_curve_path}")
print(f"Saved: {roc_curve_path}")

Saved: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/baseline_models/precision_recall_comparison.png
Saved: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/baseline_models/roc_comparison.png


/var/folders/3h/tpfh764d3msgh0j4_bys88yr0000gn/T/ipykernel_38209/1844160124.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(pr_curve_path, dpi=160, bbox_inches="tight"); plt.show(); plt.close(fig)
/var/folders/3h/tpfh764d3msgh0j4_bys88yr0000gn/T/ipykernel_38209/1844160124.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(roc_curve_path, dpi=160, bbox_inches="tight"); plt.show(); plt.close(fig)


### What the cell does
Saves the LightGBM validation metrics and fitted baseline estimator.

### Why it is necessary
Persisted artifacts make the baseline experiment reproducible without retraining and preserve failed-model information in the notebook output.

### What to understand
Only successful models are saved; their filenames identify the exact estimator represented in the comparison report.

In [8]:
comparison_path = REPORT_DIR / "baseline_model_comparison.csv"
baseline_comparison.to_csv(comparison_path, index=False)
model_filenames = {
    "DummyClassifier": "dummy_classifier.joblib",
    "LogisticRegression": "logistic_regression.joblib",
    "RandomForest": "random_forest.joblib",
    "HistGradientBoosting": "hist_gradient_boosting.joblib",
    "LightGBM": "lightgbm.joblib",
}
saved_model_paths = {}
for model_name, model in trained_models.items():
    model_path = MODEL_DIR / model_filenames[model_name]
    joblib.dump(model, model_path)
    saved_model_paths[model_name] = model_path
print(f"Saved comparison: {comparison_path}")
for model_name, model_path in saved_model_paths.items():
    print(f"Saved {model_name}: {model_path}")
if training_failures:
    print(f"Training failures: {training_failures}")

Saved comparison: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/baseline_model_comparison.csv
Saved LightGBM: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/baseline/lightgbm.joblib


### What the cell does
Extracts the 20 strongest LightGBM split importances.

### Why it is necessary
These rankings provide an initial view of which transformed inputs each fitted baseline relied on most.

### What to understand
Importance and coefficient magnitude describe model behavior, not cause and effect; correlated or engineered features can share or distort apparent importance.

In [9]:
feature_names = X_train_tree.columns.to_numpy()
importance_rows = []
if "LightGBM" in trained_models:
    values = trained_models["LightGBM"].feature_importances_.astype(float)
    for rank, index in enumerate(np.argsort(values)[::-1][:20], start=1):
        importance_rows.append({"model": "LightGBM", "importance_type": "split_importance",
                                "rank": rank, "feature_name": feature_names[index], "value": values[index]})
feature_importance = pd.DataFrame(importance_rows)
feature_importance["interpretation_note"] = "Feature importance describes model association and does not prove causality."
feature_importance_path = REPORT_DIR / "baseline_feature_importance.csv"
feature_importance.to_csv(feature_importance_path, index=False)
display(feature_importance.groupby(["model", "importance_type"], as_index=False).size())
print(f"Saved {len(feature_importance)} ranked entries to {feature_importance_path}")
print("Reminder: feature importance does not prove causality.")

,model,importance_type,size
0,LightGBM,split_importance,20


Saved 20 ranked entries to /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/baseline_feature_importance.csv
Reminder: feature importance does not prove causality.


### What the cell does
Combines validation identifiers, the true target, and every model's predicted class and probability into one row-aligned report.

### Why it is necessary
Row-level predictions support later error analysis by machine, run, segment, and time without mixing identifiers into model features.

### What to understand
Each prediction and probability belongs to the identifier row on the same line; this file is for diagnosis, not test evaluation.

In [10]:
validation_predictions = validation_identifiers.reset_index(drop=True).copy()
validation_predictions["true_target"] = y_validation.to_numpy()
for model_name, output in validation_outputs.items():
    safe_name = model_name.lower()
    validation_predictions[f"{safe_name}_predicted_class"] = output["prediction"]
    validation_predictions[f"{safe_name}_predicted_probability"] = output["probability"]
validation_predictions_path = REPORT_DIR / "baseline_validation_predictions.csv"
validation_predictions.to_csv(validation_predictions_path, index=False)
assert len(validation_predictions) == EXPECTED_VALIDATION_ROWS
print(f"Saved {len(validation_predictions):,} validation prediction rows to {validation_predictions_path}")
display(validation_predictions.head())

Saved 22,212 validation prediction rows to /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/baseline_validation_predictions.csv


,machine_id,run_id,segment_id,timestamp,valid_5min_horizon,valid_10min_horizon,slowdown_in_10min,true_target,lightgbm_predicted_class,lightgbm_predicted_probability
0,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,0890dcc046c079acc4de4202__90c048bb-46c5-4345-a...,2026-08-08T23:47:21.330000Z,True,True,0.0,0,0,0.004886
1,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,0890dcc046c079acc4de4202__90c048bb-46c5-4345-a...,2026-08-08T23:47:23.313000Z,True,True,0.0,0,0,0.009094
2,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,0890dcc046c079acc4de4202__90c048bb-46c5-4345-a...,2026-08-08T23:47:25.332000Z,True,True,0.0,0,0,0.003934
3,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,0890dcc046c079acc4de4202__90c048bb-46c5-4345-a...,2026-08-08T23:47:27.341000Z,True,True,0.0,0,0,0.003159
4,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,0890dcc046c079acc4de4202__90c048bb-46c5-4345-a...,2026-08-08T23:47:29.320000Z,True,True,0.0,0,0,0.002275


### What the cell does
Reloads saved artifacts, verifies report and model completeness, recomputes every input checksum, and prints the requested decision-oriented summary.

### Why it is necessary
Final assertions catch incomplete outputs or accidental data changes before the baseline experiment is accepted.

### What to understand
The LightGBM baseline is a provisional development reference. The different validation positive rate means its score may not generalize to future runs.

In [11]:
required_metric_columns = ["model", "preprocessing_type", "threshold", "accuracy", "balanced_accuracy",
                           "precision", "recall", "f1", "roc_auc", "pr_auc", "tn", "fp",
                           "fn", "tp", "training_time_seconds", "prediction_time_seconds"]
saved_comparison = pd.read_csv(comparison_path)
assert saved_comparison.columns.tolist() == required_metric_columns
assert saved_comparison["pr_auc"].is_monotonic_decreasing
assert all(path.exists() and path.stat().st_size > 0 for path in saved_model_paths.values())
assert all(path.exists() and path.stat().st_size > 0 for path in confusion_figure_paths + [pr_curve_path, roc_curve_path])
assert feature_importance_path.exists() and validation_predictions_path.exists()
input_hashes_after = {name: sha256_file(path) for name, path in input_paths.items()}
inputs_unchanged = input_hashes_before == input_hashes_after
assert inputs_unchanged, "At least one input file changed during baseline training."

best_pr = baseline_comparison.loc[baseline_comparison["pr_auc"].idxmax()]
best_f1 = baseline_comparison.loc[baseline_comparison["f1"].idxmax()]
best_recall = baseline_comparison.loc[baseline_comparison["recall"].idxmax()]

print("FINAL BASELINE SUMMARY")
print(f"Best by PR-AUC: {best_pr['model']} ({best_pr['pr_auc']:.4f})")
print(f"Best by F1: {best_f1['model']} ({best_f1['f1']:.4f})")
print(f"Best by recall: {best_recall['model']} ({best_recall['recall']:.4f})")
print("Missed slowdowns and false alerts at threshold 0.50:")
for row in baseline_comparison.itertuples(index=False):
    print(f"  {row.model}: missed={row.fn}, false alerts={row.fp}")
print(f"WARNING: validation positives are {y_validation.mean():.2%}, versus {y_train.mean():.2%} in training.")
print("This unusual distribution shift makes validation comparison useful but limits confidence in absolute scores.")
print("Only the requested LightGBM baseline was trained in this rerun.")
print(f"All six allowed inputs remained unchanged: {inputs_unchanged}")
print("The test split was not loaded or evaluated. No SMOTE, tuning, or train+validation refit was performed.")
if training_failures:
    print(f"Issues requiring attention: training failures={training_failures}")
else:
    print("Issues requiring attention: none; the requested LightGBM baseline trained successfully.")

FINAL BASELINE SUMMARY
Best by PR-AUC: LightGBM (0.9690)
Best by F1: LightGBM (0.8975)
Best by recall: LightGBM (0.8997)
Missed slowdowns and false alerts at threshold 0.50:
  LightGBM: missed=1095, false alerts=1147
This unusual distribution shift makes validation comparison useful but limits confidence in absolute scores.
Only the requested LightGBM baseline was trained in this rerun.
All six allowed inputs remained unchanged: True
The test split was not loaded or evaluated. No SMOTE, tuning, or train+validation refit was performed.
Issues requiring attention: none; the requested LightGBM baseline trained successfully.
